In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

In [5]:
# LOAD DATASET

df = pd.read_csv("/content/Final_MasterDataset.csv")
print("Dataset loaded successfully!")
print(df.head())
print("\nColumns:", df.columns.tolist())

Dataset loaded successfully!
  Country  Year  D_Expenditure_GDP  Conflict_Intensity  \
0     USA  1994           4.215265                 NaN   
1     USA  1995           3.860246                 NaN   
2     USA  1996           3.554982                 NaN   
3     USA  1997           3.554982                 NaN   
4     USA  1998           3.201558                 NaN   

   Health_Expenditure_(% of GDP)  Education_Expenditure_(% of GDP)  \
0                            NaN                               NaN   
1                            NaN                               NaN   
2                            NaN                               NaN   
3                            NaN                               NaN   
4                            NaN                               NaN   

   Environmental_impact(CO2e/capita)  \
0                              19.83   
1                              19.79   
2                              20.14   
3                              20.90   
4

In [6]:
# SELECT NECESSARY COLUMNS

target_col = "Health_Expenditure_(% of GDP)"   # target variable
feature_cols = ["D_Expenditure_GDP", "Year", "Country"]


In [7]:
# Filter only these columns

data = df[feature_cols + [target_col]].copy()
print("\nBefore dropping missing values:", len(data))


Before dropping missing values: 155


In [8]:
# HANDLE MISSING VALUES
# Drop rows where target or features have missing values

data = data.dropna(subset=[target_col, "D_Expenditure_GDP", "Year", "Country"])
print("After dropping missing values:", len(data))

After dropping missing values: 116


In [9]:
# Create features (X) and target (y)

X = data[feature_cols]
y = data[target_col]


In [10]:
# PREPROCESSING + MODEL PIPELINE

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Country"]),
        ("num", "passthrough", ["D_Expenditure_GDP", "Year"]),
    ]
)

In [11]:
model = Pipeline(steps=[
    ("prep", preprocess),
    ("tree", DecisionTreeRegressor(max_depth=5, random_state=42))
])

In [12]:
# TRAIN,TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
print("\nTraining rows:", len(X_train))


Training rows: 92


In [14]:
print("Testing rows:", len(X_test))

Testing rows: 24


In [15]:
# TRAIN THE MODEL

model.fit(X_train, y_train)
pred = model.predict(X_test)

In [16]:
# EVALUATE THE MODEL

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5

In [17]:
print("\n-------- MODEL PERFORMANCE --------")
print("R² Score:", round(r2, 4))
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("-----------------------------------")



-------- MODEL PERFORMANCE --------
R² Score: 0.9931
MAE: 0.2494
RMSE: 0.3439
-----------------------------------


In [18]:
# SHOW EXACT TEST SET ROWS USED FOR TESTING

test_data = X_test.copy()
test_data["Actual_Health_Expenditure"] = y_test.values
test_data["Predicted_Health_Expenditure"] = pred

print("\n-------- HEALTH EXPENDITURE TEST SET --------")
print(test_data)


-------- HEALTH EXPENDITURE TEST SET --------
     D_Expenditure_GDP  Year Country  Actual_Health_Expenditure  \
113           2.184485  2014     GBR                   9.903060   
10            4.016313  2004     USA                  14.551234   
56            3.860338  2019     RUS                   5.648491   
54            4.248996  2017     RUS                   5.358898   
16            4.904023  2010     USA                  16.197231   
69            1.983086  2001     CHN                   4.253301   
147           1.908642  2017     FRA                  11.370640   
50            3.854043  2013     RUS                   5.081049   
100           2.411122  2001     GBR                   7.452114   
17            4.822442  2011     USA                  16.140009   
59            4.690035  2022     RUS                   6.919863   
40            3.670838  2003     RUS                   5.163689   
131           2.030695  2001     FRA                   9.705628   
75            1